# RT-LCOD — End-to-End Research Notebook

Architecture implemented on this branch:

**YOLOE-26M → open-vocabulary candidates → lightweight region features → attribute/relation grounding head → target or NO_TARGET**, with **Grounding DINO used only as an offline teacher**.

The notebook is deliberately safe by default. `RUN_HEAVY = False` means **Run All** will not download YOLOE, Grounding DINO, or external datasets. It exercises parser, geometry, student forward/backward, losses, checkpointing, timing, metrics, and tracking using deterministic synthetic data.


## Session 0 — Execution mode and project root

This session locates the `rt_lcod` folder, adds `src/` to `sys.path`, and defines the heavy-model gate. Keep `RUN_HEAVY=False` until the smoke path is clean.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, HERE.parent, HERE / "rt_lcod"] if (p / "configs" / "smoke.yaml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from rt_lcod/notebooks, rt_lcod, or its parent.")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
RUN_HEAVY = False
print("RT-LCOD root:", ROOT)
print("RUN_HEAVY:", RUN_HEAVY)


## Session 1 — Environment preflight

We record Python/PyTorch/CUDA state first. Heavy package availability is reported but not treated as an error in smoke mode.


In [ ]:
import importlib.util, platform, torch
from rt_lcod.utils.logging import environment_snapshot

env = environment_snapshot()
print(env)
for package in ["torch", "torchvision", "yaml", "ultralytics", "transformers", "cv2"]:
    print(f"{package:12s}", "OK" if importlib.util.find_spec(package) else "missing/optional")


## Session 2 — Load configuration and seed

`smoke.yaml` is intentionally tiny and CPU-safe. Full experiments use `base.yaml`. All repeatable experiments start by fixing the seed.


In [ ]:
from rt_lcod.config import load_config
from rt_lcod.utils.seed import seed_everything

CONFIG_PATH = ROOT / "configs" / ("base.yaml" if RUN_HEAVY else "smoke.yaml")
config = load_config(CONFIG_PATH)
seed_everything(config.seed, deterministic=not RUN_HEAVY)
print(config)


## Session 3 — Prompt parsing contract

V1 supports one target class, at most one attribute, and at most one first-order relation/reference object. Parsing is separated from grounding so parser errors can be evaluated independently.


In [ ]:
from rt_lcod.prompt import RuleSlotParser

parser = RuleSlotParser()
examples = [
    "cup",
    "red cup",
    "red cup next to pillow",
    "bottle left of laptop",
]
for text in examples:
    print(text, "->", parser.parse(text))


## Session 4 — Geometry features for one-hop relations

The relation head does not need a large multimodal transformer. It receives candidate visual features plus normalized pairwise geometry. This makes left/right/above/below/near/overlap learnable with low per-frame cost.


In [ ]:
import torch
from rt_lcod.geometry import relation_geometry

target = torch.tensor([[0.10, 0.20, 0.30, 0.40]])
reference = torch.tensor([[0.60, 0.20, 0.80, 0.40]])
features = relation_geometry(target, reference)
print("geometry shape:", tuple(features.shape))
print("dx, dy, distance:", features[0, :3].tolist())


## Session 5 — Synthetic candidate dataset

Before touching heavy models, we verify the student contract with candidate tensors that have the same shapes as cached YOLOE outputs. This catches interface, padding, loss, and checkpoint bugs cheaply.


In [ ]:
from torch.utils.data import DataLoader
from rt_lcod.data.synthetic import SyntheticGroundingDataset
from rt_lcod.data.cached_dataset import collate_cached

dataset = SyntheticGroundingDataset(
    size=16,
    visual_dim=config.model.visual_dim,
    text_dim=config.model.text_dim,
    max_candidates=min(config.model.top_k, 6),
    seed=config.seed,
)
loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_cached)
batch = next(iter(loader))
for key, value in batch.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape), value.dtype)


## Session 6 — Student forward pass

The student scores `N candidates + NO_TARGET`. Target class candidates are gated by YOLOE labels; attribute and relation terms then resolve ambiguous same-class candidates.


In [ ]:
from rt_lcod.models.student import RTLCODStudent

student = RTLCODStudent(config.model)
output = student(batch)
print("candidate logits:", tuple(output.candidate_logits.shape))
print("full logits (includes NO_TARGET):", tuple(output.logits.shape))
print("predictions:", output.logits.argmax(dim=-1).tolist())


## Session 7 — Losses and one optimizer step

The training objective combines target selection, optional attribute supervision, optional verified relation-pair supervision, explicit no-target supervision, and optional teacher distillation.


In [ ]:
from rt_lcod.training.losses import compute_loss

optimizer = torch.optim.AdamW(student.parameters(), lr=config.training.learning_rate)
losses = compute_loss(output, batch, config.loss, config.training.kd_temperature)
optimizer.zero_grad(set_to_none=True)
losses.total.backward()
optimizer.step()
print(losses.scalar_dict())


## Session 8 — Checkpoint/resume and run tracking

A research run must be resumable. Checkpoints contain model, optimizer, scheduler, scaler, epoch/global step, best metric, and RNG state. Run folders also store environment, config, metrics, and timing.


In [ ]:
import tempfile
from rt_lcod.training.checkpoint import save_checkpoint, load_checkpoint

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1)
with tempfile.TemporaryDirectory(prefix="rtlcod-notebook-") as tmp:
    ckpt = Path(tmp) / "last.pt"
    save_checkpoint(
        ckpt,
        model=student,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=None,
        epoch=0,
        global_step=1,
        best_metric=0.0,
        config={"notebook": True},
    )
    restored = RTLCODStudent(config.model)
    state = load_checkpoint(ckpt, model=restored, restore_rng=False)
    print("checkpoint restored:", {k: state[k] for k in ["epoch", "global_step", "best_metric"]})


## Session 9 — Temporal tracker smoke test

Tracking is a deployment optimization, not a replacement for LCOD evaluation. Raw model FPS and practical tracked-system FPS must be reported separately.


In [ ]:
from rt_lcod.inference.temporal import SimpleTargetTracker

tracker = SimpleTargetTracker(iou_gate=0.2, max_missed=2)
print(tracker.update([((10, 10, 30, 30), 0.90)]))
print(tracker.update([((12, 11, 32, 31), 0.85)]))
print(tracker.update([]))


## Session 10 — Heavy Stage 0/1 preparation (gated)

When `RUN_HEAVY=True`, the next commands are the real path:

1. validate canonical manifests;
2. run YOLOE-26M to cache target/reference candidates;
3. compute frozen MobileNetV3 region features;
4. cache text embeddings;
5. train the grounding head.

This notebook prints the commands instead of silently downloading models.


In [ ]:
commands = [
    "python scripts/validate_manifest.py --manifest data/manifests/train.jsonl",
    "python scripts/cache_candidates.py --config configs/base.yaml --manifest data/manifests/train.jsonl --output data/candidate_cache/train --model yoloe-26m.pt",
    "python scripts/cache_candidates.py --config configs/base.yaml --manifest data/manifests/val.jsonl --output data/candidate_cache/val --model yoloe-26m.pt",
    "python scripts/train.py --config configs/base.yaml --train-cache data/candidate_cache/train --val-cache data/candidate_cache/val --run-name stage1",
]
print("Heavy mode is", RUN_HEAVY)
for command in commands:
    print("  ", command)


## Session 11 — Teacher distillation (gated)

Grounding DINO is an **offline teacher only**. Teacher predictions are cached once, aligned to student candidates, and converted to a distribution over candidates plus NO_TARGET. Student inference remains teacher-free.


In [ ]:
teacher_commands = [
    "python scripts/cache_teacher.py --manifest data/manifests/train.jsonl --candidate-cache data/candidate_cache/train --output data/teacher_cache/train",
    "python scripts/train.py --config configs/base.yaml --train-cache data/candidate_cache/train --val-cache data/candidate_cache/val --teacher-cache data/teacher_cache/train --resume runs/<stage1>/checkpoints/best.pt --run-name stage2_kd",
]
for command in teacher_commands:
    print("  ", command)


## Session 12 — Evaluation and real-time benchmark

Do not declare a model real-time from average FPS alone. Record mean/p50/p95/p99 latency, deadline misses, VRAM, and prompt-change latency. Evaluate class-only, attribute, relation, combined, and no-target subsets separately.


In [ ]:
print("Evaluation:")
print("  python scripts/evaluate.py --config configs/base.yaml --cache data/candidate_cache/val --checkpoint runs/<run>/checkpoints/best.pt")
print("Runtime:")
print('  python scripts/benchmark_runtime.py --config configs/base.yaml --checkpoint runs/<run>/checkpoints/best.pt --video ../data/videos/highway_traffic_gpl.mp4 --prompt "red cup next to pillow"')


## Session 13 — Stage gates

Before advancing:

- **Proposal gate:** YOLOE must actually propose the target/reference objects.
- **Grounding gate:** the head must beat highest-detector-confidence selection on ambiguous scenes.
- **Negative gate:** no-target false positives must be measured.
- **KD gate:** distillation must improve accuracy without runtime cost.
- **Realtime gate:** p95 latency must be measured on the target robot GPU.
- **Robot gate:** perception correctness must be separated from grasp/manipulation correctness.

The full rationale, datasets, losses, hard-negative strategy, training stages, and robot deployment plan are in `../../plan.md`.
